# LangGraph RAG Agent over pgvector

This is **Part B** of a two-part notebook. It assumes the ingestion pipeline in
**`11a.rag_pipeline.ipynb`** has already run and populated the pgvector collection
`langgraph_rag_demo`. Here we simply **connect to that existing vector store** and build a
**Retrieval-Augmented Generation (RAG) agent** with LangGraph:

1. **Connect** — attach to the existing pgvector collection (no re-embedding)
2. **Retrieval tool** — wrap the retriever so the agent can call it
3. **Agent** — a LangGraph agent decides when to retrieve and synthesizes answers
4. **Persistence** — conversation memory via PostgresSaver

```mermaid
graph LR
    PG["🐘 pgvector<br/>(langgraph_rag_demo)"] -->|retriever tool| Agent["🤖 LangGraph Agent"]
    Agent -->|PostgresSaver| Memory["💾 Persistent Memory"]
```

**What makes this different from a basic RAG chain:**
- The agent decides whether retrieval is needed (not every question requires it)
- It can search multiple times with different queries to gather more context
- The graph is persistent — uses PostgresSaver so conversations survive restarts

**Prerequisites:**
- **Run `11a.rag_pipeline.ipynb` first** to build the pgvector collection
- pgvector pod running (`podman pod start pgvector-pod`)
- MLflow UI running (`mlflow ui --port 5000`)
- `.env` file configured

## Step 1 — Bootstrap & Connect

We run the same shared configuration script as Part A. Using the identical setup is what allows this
notebook to reach the exact database and embedding model that Part A wrote to.

**What this establishes:**
- **The same embedding model and pgvector connection** used during ingestion — essential, because a
  query must be embedded with the *same* model as the stored chunks for similarity to be meaningful.
- **The `PGVector` interface** to attach to the existing collection.
- **The LLM client and helpers** used to build and run the agent later in this notebook.

In [1]:
%run ../langchain_common.py

C:\Users\akhawaja\git\cs4603\langchain_common.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
USER_AGENT environment variable not set, consider setting it to identify your requests.


## Step 2 — Connect to the Existing pgvector Collection

This is where Part B picks up exactly where **Part A (`11a.rag_pipeline.ipynb`)** left off. All the
expensive work — converting PDFs, chunking, and embedding — already happened there and was saved to
pgvector. Here we simply **attach** to that same collection.

**The logic of this stage:**
- **Reuse, don't rebuild.** We open the existing collection instead of re-ingesting, so no PDFs are
  read and nothing is re-embedded. This is the whole point of separating ingestion from serving:
  embed once, then reuse across many sessions.
- **Fail fast if it's empty.** A quick similarity search doubles as a guard — if the collection has
  no data, it means Part A hasn't been run yet, and the notebook stops with a clear message instead
  of failing mysteriously later.

**Why this separation matters:**
- Ingestion is slow and only needs to happen when documents change; serving happens constantly.
  Keeping them apart makes this notebook fast to start and cheap to re-run.

In [2]:
COLLECTION_NAME = "langgraph_rag_demo"

# Attach to the collection that 11a already populated (no re-embedding).
vectorstore = PGVector(
    embeddings=databricks_embeddings,
    collection_name=COLLECTION_NAME,
    connection=pgvectordb_conn,
    use_jsonb=True,
    pre_delete_collection=False,  # reuse existing vectors
)

# Sanity check — make sure the collection actually has data.
test_results = vectorstore.similarity_search_with_score("What is RAG?", k=2)
if not test_results:
    raise RuntimeError(
        f"Collection '{COLLECTION_NAME}' is empty. "
        "Run 11a.rag_pipeline.ipynb first to build the vector store."
    )

print(f"Connected to '{COLLECTION_NAME}'. Sample matches:")
for doc, score in test_results:
    source = doc.metadata.get("source", "unknown")
    print(f"  score={score:.4f} | ({source}) {doc.page_content[:80]}...")

Connected to 'langgraph_rag_demo'. Sample matches:
  score=0.3209 | (2005.11401v4 Retrieval-Augmented Generation.pdf) 2 Methods

We explore RAG models, which use the input sequence x to retrieve tex...
  score=0.3237 | (2005.11401v4 Retrieval-Augmented Generation.pdf) Retrieve-and-Edit approaches Our method shares some similarities with retrieve-a...


## Step 3 — Define the Retrieval Tool

For the agent to *decide* when to look something up, retrieval has to be exposed as a **tool** it can
choose to call — not a step that always runs.

**The logic of this stage:**
- **Wrap the vector store as a callable tool.** The tool takes a search query, embeds it, and returns
  the most similar chunks from the collection that Part A built.
- **Describe the tool clearly.** The tool's description tells the agent *what* it can look up and
  *when* to use it. The model relies on this description to decide whether a given question warrants
  a search.
- **Return results with their sources.** Each retrieved passage is labeled with the paper it came
  from, so the agent can ground its answer and cite where the information originated.

**Why it's a tool and not a fixed step:** a hard-wired retrieve-then-answer chain searches for
*every* input, even a greeting. Making retrieval a tool lets the agent skip it when it isn't needed
and call it repeatedly when a question needs more context.

In [3]:
from langchain_core.tools import tool

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


@tool
def search_knowledge_base(query: str) -> str:
    """Search the knowledge base of AI research papers for information about
    LLMs, RAG, multi-agent systems, transformers, and attention mechanisms.
    Use this tool when the user asks factual questions about these topics.

    Args:
        query: the search query describing what information you need
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant documents found in the knowledge base."

    results = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "unknown")
        results.append(f"[{i}] (source: {source})\n{doc.page_content}")

    return "\n\n".join(results)


# Verify the tool works
print(search_knowledge_base.invoke("What is retrieval augmented generation?")[:500])

[1] (source: 2308.08155v2 Autogen - Enabling Next-Gen LLM Apps via Multiagent Conversations.pdf)
A2: Retrieval-Augmented Code Generation and Question Answering

Retrieval augmentation has emerged as a practical and effective approach for mitigating the intrinsic
limitations of LLMs by incorporating external documents. In this section, we employ AutoGen to
build a Retrieval-Augmented Generation (RAG) system (Lewis et al., 2020; Parvez et al., 2021)
named Retrieval-augmented Chat. The system consi


## Step 4 — Build the LangGraph RAG Agent

Now we assemble the agent as a small **graph**: the model can either answer directly or call the
retrieval tool, loop back with what it found, and try again until it's ready to respond.

**The logic of this stage:**
- **A system prompt sets the policy.** It instructs the model to search the knowledge base for
  factual questions, to ground answers in what it retrieves, and to answer directly for things that
  don't need retrieval (greetings, arithmetic, opinions).
- **Two nodes, connected in a loop.** One node is the model's turn; the other runs any tool calls it
  requested. After a tool runs, control returns to the model so it can use the results — and it may
  search again with a refined query before finally answering.
- **Automatic routing.** A conditional edge inspects the model's output: if it asked for a tool, the
  graph runs the tool; if not, the graph ends with the answer. No manual if/else orchestration.
- **Persistent memory.** The graph is compiled with a Postgres-backed checkpointer, so conversation
  state is saved and can survive kernel restarts.

**The key idea:** the control flow is *decided by the model at runtime*, not fixed in advance — that
adaptivity is what separates an agent from a static pipeline.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.messages import HumanMessage, SystemMessage

RAG_SYSTEM_PROMPT = """You are a knowledgeable AI research assistant with access to a \
knowledge base of research papers about Large Language Models, Retrieval-Augmented \
Generation, multi-agent systems, and transformer architectures.

Rules:
1. When the user asks factual questions about LLMs, RAG, transformers, attention, \
   multi-agent systems, or related AI topics, ALWAYS use the search_knowledge_base \
   tool first to find relevant information from the papers.
2. You may search multiple times with different queries to gather comprehensive context.
3. Base your answers on the retrieved documents. Cite the source paper when possible.
4. If the knowledge base doesn't have relevant information, say so honestly and provide \
   your best general knowledge answer.
5. For non-factual requests (greetings, math, opinions), respond directly without searching.
"""

tools = [search_knowledge_base]
llm_with_tools = llm_noreason.bind_tools(tools)


def assistant(state: MessagesState):
    messages = [SystemMessage(content=RAG_SYSTEM_PROMPT)] + state["messages"]
    return {"messages": [llm_with_tools.invoke(messages)]}


# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

# Compile with persistent Postgres checkpointer
pg_checkpointer = create_pg_checkpointer()
graph = builder.compile(checkpointer=pg_checkpointer)

print("RAG agent compiled with pgvector retrieval + Postgres checkpointer")

## Step 5 — Test the Agent

We probe the agent with a mix of questions to confirm it makes the retrieve-or-not decision
correctly.

**What each test is checking:**
- **A factual question** that lives in the papers — the agent should call the retrieval tool and
  ground its answer in what it finds.
- **A cross-paper question** — the agent may search more than once, with different queries, to gather
  context spanning multiple documents.
- **A non-factual question** (simple arithmetic) — the agent should recognize retrieval isn't needed
  and answer directly, skipping the knowledge base entirely.

All three run on the **same conversation thread**, so the agent also carries context forward from one
turn to the next.

In [ ]:
# Start a conversation thread
config = make_thread_config()
print(f"Thread: {config['configurable']['thread_id']}\n")

# Question 1: Requires retrieval (from the RAG paper)
result = graph.invoke(
    {"messages": [HumanMessage(content="How does RAG combine retrieval with generation? What architecture does the original paper propose?")]},
    config=config,
)
print("Q: How does RAG combine retrieval with generation?")
print(f"A: {result['messages'][-1].content}\n")

In [ ]:
# Question 2: Cross-paper question (AutoGen + PagedAttention)
result = graph.invoke(
    {"messages": [HumanMessage(content="What is PagedAttention and how does it improve LLM serving efficiency?")]},
    config=config,
)
print("Q: What is PagedAttention and how does it improve LLM serving efficiency?")
print(f"A: {result['messages'][-1].content}\n")

In [ ]:
# Question 3: Does NOT require retrieval — agent should answer directly
result = graph.invoke(
    {"messages": [HumanMessage(content="What is 42 * 17?")]},
    config=config,
)
print("Q: What is 42 * 17?")
print(f"A: {result['messages'][-1].content}")
print("\n(The agent should answer directly without searching the knowledge base)")

## Step 6 — Inspect the Agent's Reasoning

Printing the final answer hides *how* the agent got there. Here we look at the full sequence of
messages for one query to make the agent's decision process visible.

**What the trace reveals:**
- **The user's question**, followed by the model's choice to call the retrieval tool (including the
  exact search query it invented).
- **The tool's output** — the passages that came back from pgvector.
- **The model's final answer**, synthesized from those passages.

Seeing this end-to-end makes the "decide → retrieve → read → answer" loop concrete, and is invaluable
when debugging *why* an agent retrieved (or failed to retrieve) certain information.

In [ ]:
# New thread for a clean trace
trace_config = make_thread_config()

result = graph.invoke(
    {"messages": [HumanMessage(content="How does AutoGen enable multi-agent conversations and what design patterns does it use?")]},
    config=trace_config,
)

print("=" * 60)
print("Full message trace:")
print("=" * 60)
for msg in result["messages"]:
    msg.pretty_print()
    print()

## Step 7 — Verify Persistence

Because the graph was compiled with a Postgres-backed checkpointer, conversation state isn't held only
in memory — it's saved. This step demonstrates that durability.

**The logic of this stage:**
- **Reload a past conversation by its thread id.** We fetch the stored state for an earlier thread and
  confirm its messages are still there, even though nothing was kept in local variables.
- **Continue where it left off.** Sending a follow-up on that same thread shows the agent still has the
  full history and can summarize everything discussed.

**Why this matters:** persistent memory is what turns a one-off script into something that behaves like
a real assistant — conversations survive restarts, and each thread keeps its own independent history.

In [ ]:
# Retrieve the state from the first conversation thread
state = graph.get_state(config)
print(f"Thread: {config['configurable']['thread_id']}")
print(f"Messages in thread: {len(state.values['messages'])}")
print(f"\nLast message: {state.values['messages'][-1].content[:200]}...")

# Continue the conversation
result = graph.invoke(
    {"messages": [HumanMessage(content="Can you summarize everything we discussed?")]},
    config=config,
)
print(f"\nSummary: {result['messages'][-1].content}")

## Summary

This notebook consumed the pgvector collection built in **`11a.rag_pipeline.ipynb`** and put a
LangGraph agent on top of it.

### Agent Architecture

| Component | Technology | What it does |
|-----------|-----------|--------------|
| Vector store | **pgvector** (`PGVector`) | Existing collection from Part A — attached, not rebuilt |
| Retrieval tool | `@tool` decorator | Wraps the pgvector retriever for agent tool calling |
| Agent graph | **LangGraph** (`StateGraph`) | Decides when to retrieve, synthesizes answers |
| Persistence | **PostgresSaver** | Conversation memory survives restarts |

### Key Takeaways

- Ingestion (Part A) and serving (Part B) are **decoupled** — you embed once, then reuse the
  vector store across many agent sessions without re-running the pipeline
- The agent _chooses_ when retrieval is needed — for greetings or math it responds directly
- It can search multiple times with refined queries before composing its answer
- **PostgresSaver** means the conversation state persists across kernel restarts